In [0]:
# Silver Quality Test Script
# English comments for clarity

# Import necessary functions - This solves the NameError
from pyspark.sql.functions import col

# Load the newly created silver table
df_silver = spark.table("workspace.silver.cleaned_yellow_taxi")

# 1. Check for Nulls in critical business columns
# We expect 0 nulls in pickup time and total amount
critical_nulls = df_silver.filter(
    col("tpep_pickup_datetime").isNull() | 
    col("total_amount").isNull()
).count()

# 2. Financial Integrity Check
# Ensure fare_amount is not negative (handled by our 'when' clause earlier)
negative_fares = df_silver.filter(col("fare_amount") < 0).count()

# 3. Row Count Verification
# Check if we still have our ~47 million rows
silver_count = df_silver.count()
print(f"Total rows in Silver: {silver_count}")

# Validation Results
if critical_nulls == 0 and negative_fares == 0:
    print("Success: Data quality checks passed!")
else:
    # Raising an exception will stop the Databricks Job/Orchestration
    raise Exception(f"Quality Check Failed: Found {critical_nulls} nulls and {negative_fares} negative fares.")